In [2]:
import sys
sys.path.insert(0,'/mnt/AEA8F340A8F3059D/sportsbet/ai-engine')

In [5]:
from agents.memory import loadMemory,saveMemory
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from config.settings import settings
import json,logging

In [6]:
logger=logging.getLogger(__name__)

In [7]:
CHAT_SYSTEM_PROMPT="""You are SportsBet AI, a friendly and knowledgeable cricket and sports betting assistant.

You help users with:
- Cricket match analysis, player stats, team form
- Betting advice, odds interpretation, EV calculations
- Stock trading advice for player/team stocks
- General cricket questions and trivia

CURRENT PAGE CONTEXT:
{page_context}

USER PROFILE:
{user_context}

RELEVANT KNOWLEDGE:
{rag_context}

Guidelines:
- Be concise but thorough
- Use cricket terminology naturally
- When giving betting advice, always mention risk
- Reference specific data when available
- Be enthusiastic about cricket!
- Format responses with markdown when helpful
"""

In [1]:
async def chatAssistant(state):
    query=state.get("query","")
    userId=state.get("user-id","")
    ctx=state.get("context",{})
    sessionId=state.get("sessionId","default")

    historyMessages=await loadMemory(sessionId,limit=settings.CHAT_HISTORY_LIMIT)
    llm=ChatGoogleGenerativeAI(
        model=settings.GEMINI_MODEL,
        google_api_key=settings.GEMINI_API_KEY,
        temperature=0.7
    )
    prompt=ChatPromptTemplate.from_messages([
        ("system",CHAT_SYSTEM_PROMPT),
        MessagesPlaceholder(variable_name="history"),
        ("human","{query}")
    ])
    chain=prompt|llm
    try:
        result=await chain.ainvoke({
            "page-context":json.dumps(ctx,default=str),
            "user-context":json.dumps({
                "profile":state.get("user-profile",{}),
                "wallet":state.get("user-wallet",{}),
            },default=str)[:1000],
            "rag-context":state.get("chunks",[]),
            "history":historyMessages[-20:],
            "query":query
        })
        response=result.content if isinstance(result.content,str) else ""
    except Exception as e:
        logger.error(f"chat failed {e}")
        response="Trouble connecting to the AI service"
    if userId and userId!="system":
        await saveMemory(sessionId,userId,"user",query,ctx)
        await saveMemory(sessionId,userId,"assistant",response,ctx)
    
    return{
        "output":{
            "response":response,
            "sessionId":sessionId
        }
    }

In [1]:
async def chatAssistantStream(state):
    query=state.get("query","")
    userId=state.get("user-id","")
    ctx=state.get("context",{})
    sessionId=ctx.get("sessionId","default")
    historyMessages=await loadMemory(
        sessionId,
        limit=10
    )
    llm=ChatGoogleGenerativeAI(
            model=settings.GEMINI_MODEL,
            google_api_key=settings.GEMINI_API_KEY,
            temperature=0.7,
            streaming=True
        )
    prompt=ChatPromptTemplate.from_messages([
        ("system",CHAT_SYSTEM_PROMPT),
        MessagesPlaceholder(variable_name="history"),
        ("human","{query}")
    ])
    chain=prompt|llm
    fullResponse=""
    async for chunk in chain.astream({
        "page-context":json.dumps(ctx,default=str),
        "user-context":json.dumps({
            "profile":state.get("user-profile",{}),
            "wallet":state.get("user-wallet",{}),
        },default=str)[:1000],
        "rag-context":state.get("chunks",[]),
        "history":historyMessages[-20:],
        "query":query
    }):
        token=chunk.content if isinstance(chunk.content,str) else ""
        if token:
            fullResponse+=token
            yield token
    if userId and userId!="system":
        await saveMemory(sessionId,userId,"user",query,ctx)
        await saveMemory(sessionId,userId,"assistant",fullResponse,ctx)